<a href="https://colab.research.google.com/github/IshiJess/BITACORA-DE-SECUENCIACION--Jessica-Noemi-Salcedo-Alferez-CaBioinf2027/blob/main/Bitacora_Metagenomica_Shotgun.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:#A230A4;">**Metagenómica Shotgun (de Novo y Binning)**</span>

**Estudiante:** Jessica Noemi Salcedo Alferez  
**Fecha:** Noviembre 2026  
**Entorno de ejecución:** Google colab/ Linux Bash o Ubuntu sobre windows/ Rstudio (Phyloseq)/ GitHub

**Proyecto Base:** Metagenómica Shotgun en Pozas de Cuatro Ciénegas, Coahuila, México


## Introducción y Contexto Experimental

La **metagenómica Shotgun** consiste en la secuenciación masiva y aleatoria de todo el ADN genómico presente en una muestra ambiental. A diferencia del metabarcoding (amplicón 16S), la metagenómica Shotgun permite:
1. Identificar la composición taxonómica (bacterias, arqueas, virus, eucariotas).
2. Caracterizar el potencial metabólico y funcional completo de la comunidad.
3. Reconstruir Genomas Ensamblados desde Metagenomas (**MAGs**).

## FASE 1: Configuración del Entorno en Google Colab

In [ ]:
# 1. Verificar espacio de trabajo y memoria RAM disponible
!free -h
!df -h

# 2. Instalación de Conda en Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()

# 3. Instalación de herramientas bioinformáticas esenciales
!conda create -n metagenomics -c bioconda -c conda-forge fastqc trimmomatic metaspades maxbin2 checkm-genome kraken2 phyloseq r-ggplot2 r-patchwork -y

## FASE 2: Control de Calidad y Preprocesamiento de Lecturas Crudas

In [ ]:
# 1. Creación de directorios y control de calidad inicial
!mkdir -p data/untrimmed_fastq results/fastqc_untrimmed

# Descarga / Acceso a datos pareados (JC1A: Control mesocosmo | JP4D: Poza fertilizada)
!fastqc data/untrimmed_fastq/*.fastq.gz -o results/fastqc_untrimmed/

# 2. Recorte de adaptadores y filtrado de calidad (Trimmomatic)
!mkdir -p data/trimmed_fastq
!trimmomatic PE -phred33 \
  data/untrimmed_fastq/JC1A_R1.fastq.gz data/untrimmed_fastq/JC1A_R2.fastq.gz \
  data/trimmed_fastq/JC1A_R1.trim.fastq.gz data/trimmed_fastq/JC1A_R1.unpaired.fastq.gz \
  data/trimmed_fastq/JC1A_R2.trim.fastq.gz data/trimmed_fastq/JC1A_R2.unpaired.fastq.gz \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

## FASE 3: Ensamble Metagenómico de Novo (MetaSPAdes)

In [ ]:
# Ensamble de novo con MetaSPAdes basado en Grafos de De Bruijn (k-mers múltiples: 21, 33, 55)
!mkdir -p results/assembly_JC1A

!metaspades.py \
  -1 data/trimmed_fastq/JC1A_R1.trim.fastq.gz \
  -2 data/trimmed_fastq/JC1A_R2.trim.fastq.gz \
  -o results/assembly_JC1A/

# Verificación de los archivos generados: contigs.fasta y scaffolds.fasta
!ls -lh results/assembly_JC1A/

## FASE 4: Binning Metagenómico y Reconstrucción de MAGs (MaxBin 2.0)

In [ ]:
# Agrupamiento de contigs en Bins basados en cobertura de lecturas y frecuencias de tetranucleótidos
!mkdir -p results/assembly_JP4D/MAXBIN

!run_MaxBin.pl \
  -thread 4 \
  -contig results/assembly_JP4D/JP4D_contigs.fasta \
  -reads data/trimmed_fastq/JP4D_R1.trim.fastq.gz \
  -reads2 data/trimmed_fastq/JP4D_R2.trim.fastq.gz \
  -out results/assembly_JP4D/MAXBIN/JP4D

# Revisar resumen de Bins obtenidos
!cat results/assembly_JP4D/MAXBIN/JP4D.summary

## FASE 5: Control de Calidad de MAGs (CheckM)

In [ ]:
# Evaluación de Completitud (Completeness) y Contaminación (Contamination) de los MAGs
!mkdir -p results/CHECKM

!checkm taxonomy_wf domain Bacteria \
  -x fasta \
  results/assembly_JP4D/MAXBIN/ \
  results/CHECKM/

# Exportación de reporte detallado de calidad con métricas N50 y L50
!checkm qa results/CHECKM/Bacteria.ms results/CHECKM/ \
  --file results/CHECKM/quality_JP4D.tsv \
  --tab_table -o 2

## FASE 6: Asignación Taxonómica (Kraken 2 y kraken-biom)

In [ ]:
# Clasificación taxonómica basada en exact k-mer matching
!mkdir -p taxonomy

!kraken2 --db minikraken_db \
  --paired data/trimmed_fastq/JP4D_R1.trim.fastq.gz data/trimmed_fastq/JP4D_R2.trim.fastq.gz \
  --output taxonomy/JP4D.kraken \
  --report taxonomy/JP4D.report

# Conversión de reportes de Kraken a formato BIOM estandarizado
!kraken-biom taxonomy/JC1A.report taxonomy/JP4D.report taxonomy/JP41.report \
  --fmt json \
  -o taxonomy/cuatroc.biom

## FASE 7: Análisis de Diversidad Alfa, Beta y Abundancia Taxonómica en R (Phyloseq)

In [ ]:
%%writefile script_phyloseq.R
# Cargar librerías necesarias en R
library(phyloseq)
library(ggplot2)
library(RColorBrewer)
library(patchwork)

# 1. Importar archivo BIOM y construcción del objeto Phyloseq
merged_metagenomes <- import_biom("taxonomy/cuatroc.biom")

# Limpieza de etiquetas taxonómicas
merged_metagenomes@tax_table@.Data <- substring(merged_metagenomes@tax_table@.Data, 4)
colnames(merged_metagenomes@tax_table@.Data) <- c("Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species")

# 2. Análisis de Diversidad Alfa (Chao1, Shannon, Observed)
merged_bacteria <- subset_taxa(merged_metagenomes, Kingdom == "Bacteria")
pdf("diversidad_alfa.pdf")
plot_richness(merged_bacteria, measures = c("Observed", "Chao1", "Shannon"), title = "Diversidad Alfa en Muestras de Cuatro Ciénegas")
dev.off()

# 3. Transformación a Abundancia Relativa (%)
percentages <- transform_sample_counts(merged_bacteria, function(x) x * 100 / sum(x))

# 4. Diversidad Beta (NMDS con Distancia Bray-Curtis)
meta_ord <- ordinate(percentages, method = "NMDS", distance = "bray")
pdf("diversidad_beta.pdf")
plot_ordination(percentages, meta_ord, title = "Diversidad Beta (NMDS Bray-Curtis)")
dev.off()

# 5. Gráfico de Abundancia Relativa a Nivel de Phylum
phylum_glom <- tax_glom(percentages, taxrank = "Phylum")
phylum_df <- psmelt(phylum_glom)
phylum_df$Phylum[phylum_df$Abundance < 0.5] <- "Phyla < 0.5% abund."

pdf("abundancia_phylum.pdf")
ggplot(phylum_df, aes(x = Sample, y = Abundance, fill = Phylum)) +
  geom_bar(stat = "identity", position = "stack") +
  theme_minimal() +
  labs(title = "Composición Taxonómica por Phylum", y = "Abundancia Relativa (%)")
dev.off()

In [ ]:
# Ejecutar script de R desde Colab
!Rscript script_phyloseq.R

## Resumen Metodológico
- **Control de Calidad (FastQC/Trimmomatic):** Eliminación de lecturas de baja calidad y adaptadores.
- **Ensamble (MetaSPAdes):** Reconstrucción de contigs y scaffolds mediante Grafos de De Bruijn.
- **Binning (MaxBin 2.0):** Agrupación de contigs en MAGs por abundancia y tetranucleótidos.
- **Calidad de MAGs (CheckM):** Evaluación de completitud y contaminación.
- **Análisis de Diversidad (Phyloseq en R):** Cuantificación de diversidad alfa, beta y abundancia relativa.